# Werewolf Transformer — iPad / Google Colab smoke run

iPad は操作端末として使い、学習計算は Colab の GPU で実行します。
このノートブックは **最初の128村**だけを回します。結果は Google Drive に保存され、中断後も再実行すれば自動で resume します。

### あなたが行うこと
1. Colab でこのノートブックを開く
2. **ランタイム → ランタイムのタイプを変更 → GPU** を選ぶ
3. 上から順番にセルを実行する
4. 最後に表示される `LAST METRICS` を ChatGPT に送る

戦術ルールや報酬は変更しません。自己対戦は既存の sparse terminal reward `+1 / -1 / 0` のままです。


In [ ]:
# 1) Google Drive を接続（学習状態を永続化）
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) 最新 main を取得して依存関係をインストール
%cd /content
!rm -rf Are-you-werewolf
!git clone --depth 1 https://github.com/dolphin23-jp/Are-you-werewolf.git
%cd /content/Are-you-werewolf/backend
!python -m pip install -q -e ".[rl,transformer]"


In [ ]:
# 3) GPU と保存先を確認
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU が有効ではありません。Colab の「ランタイム → ランタイムのタイプを変更」で GPU を選んでから、最初から実行してください。'
    )

RUN_ROOT = Path('/content/drive/MyDrive/werewolf-training/pilot-001')
(RUN_ROOT / 'bootstrap').mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'pool').mkdir(parents=True, exist_ok=True)

print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('保存先:', RUN_ROOT)


In [ ]:
# 4) 128村 smoke run。既存 run-state があれば自動 resume。
import subprocess

bootstrap = RUN_ROOT / 'bootstrap'
pool_dir = RUN_ROOT / 'pool'
model = bootstrap / 'model.npz'
run_state = bootstrap / 'run.npz'
metrics = bootstrap / 'metrics.jsonl'

common = [
    'python', 'scripts/train_self_play_torch.py',
    '--episodes', '128',
    '--pool-dir', str(pool_dir),
    '--output', str(model),
    '--run-state', str(run_state),
    '--metrics-jsonl', str(metrics),
    '--device', 'auto',
]

if run_state.exists():
    cmd = common + ['--resume']
    print('既存の学習状態を検出したため resume します。')
else:
    cmd = common + [
        '--batch-size', '32',
        '--parallel-games', '8',
        '--inference-batch-size', '64',
        '--seed', '1001',
    ]
    print('新しい128村 smoke runを開始します。')

subprocess.run(cmd, check=True)
print('\nSMOKE RUN COMPLETE')


In [ ]:
# 5) ChatGPT に送る結果を表示
import json

metrics = RUN_ROOT / 'bootstrap' / 'metrics.jsonl'
if not metrics.exists():
    raise RuntimeError('metrics.jsonl がまだありません。直前のセルのエラーを確認してください。')

rows = [json.loads(line) for line in metrics.read_text().splitlines() if line.strip()]
print('completed batches:', len(rows))
print('\n===== LAST METRICS =====')
print(json.dumps(rows[-1], ensure_ascii=False, indent=2))
print('===== END LAST METRICS =====')

print('\nこの LAST METRICS をそのまま ChatGPT に貼り付けてください。')


## 途中で切れた場合

慌てなくて大丈夫です。Google Drive に `run.npz` が残っています。
新しい Colab セッションでこのノートブックを再度上から実行すると、4番目のセルが自動で `--resume` を選びます。

128村が完了したら、**ここではまだ population iteration を開始せず**、最後の metrics を ChatGPT に送ってください。速度・KL・entropy・value explained variance・GPU batch効率を確認してから次へ進みます。
